In [ ]:
pip install -r requirements.txt

## Polarisation Assessment

In [32]:
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.preprocessing import StandardScaler
import os

# Load the encoded dataset
df_encoded = pd.read_csv('../data/processed/wvs_encoded.csv')

# From the processed folder load and read the dataset for the four waves
df_wave4 = pd.read_csv('../data/processed/wvs_wave4.csv')
df_wave5 = pd.read_csv('../data/processed/wvs_wave5.csv')
df_wave6 = pd.read_csv('../data/processed/wvs_wave6.csv')
df_wave7 = pd.read_csv('../data/processed/wvs_wave7.csv')

## Running LDA with Fixed Ideology Types to 4

In [ ]:
import os
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
import joblib

# Ensure reports folder exists
output_dir = "reports/top_issues"
os.makedirs(output_dir, exist_ok=True)

# Loop over waves
for wave_num in waves:
    print(f"\nProcessing Wave {wave_num} with K={fixed_k}")
    
    df = wave_data[wave_num]
    
    # Drop metadata columns
    lda_data = df.drop(columns=["country", "year"])
    
    # Fit LDA model
    lda_model = LatentDirichletAllocation(
        n_components=fixed_k,
        doc_topic_prior=0.25,
        topic_word_prior=0.1,
        learning_method='online',
        learning_decay=0.7,
        learning_offset=10.0,
        max_iter=20,
        batch_size=1000,
        evaluate_every=-1,
        mean_change_tol=0.001,
        max_doc_update_iter=100,
        n_jobs=-1,
        random_state=25
    )

        
    lda_model.fit(lda_data)

    # After fitting LDA for each wave
    joblib.dump(lda_model, f"models/lda_model_wave{wave_num}_k4.pkl")
    
    # Extract and normalize topic-word matrix
    topic_words = pd.DataFrame(lda_model.components_, columns=lda_data.columns)
    topic_words = topic_words.div(topic_words.sum(axis=1), axis=0)
    topic_words = topic_words.T
    topic_words.columns = [f"Ideology_{i+1}" for i in range(fixed_k)]
    
    # Top 10 issues per ideology type
    top_issues = topic_words.apply(lambda x: x.nlargest(10).index.tolist(), axis=0)
    
    # Raw outputs to reports folder
    top_issues.to_csv(os.path.join(output_dir, f"wave{wave_num}_top_issues_k4.csv"), index=False)

    
    print(f"✅ Saved top 10 issues for each ideology in Wave {wave_num}")


Processing Wave 4 with K=4
✅ Saved top 10 issues for each ideology in Wave 4

Processing Wave 5 with K=4
✅ Saved top 10 issues for each ideology in Wave 5

Processing Wave 6 with K=4
✅ Saved top 10 issues for each ideology in Wave 6

Processing Wave 7 with K=4
✅ Saved top 10 issues for each ideology in Wave 7


In [36]:
# For a given topic_words DataFrame (issue x topic β-values)
top_issues = topic_words.apply(lambda x: x.nlargest(10).index.tolist(), axis=0)

In [37]:
import json

# Load your variable dictionary
with open("variable_dict.json", "r") as f:
    variable_dict = json.load(f)

# Extend the dictionary to include _support and _oppose
extended_dict = {}
for var, desc in variable_dict.items():
    extended_dict[f"{var}_support"] = f"{desc} (Support)"
    extended_dict[f"{var}_oppose"] = f"{desc} (Oppose)"

# Function to apply the mapping to a top_issues DataFrame
def map_features_to_labels(top_issues_df):
    return top_issues_df.applymap(lambda x: extended_dict.get(x, x))  # fallback to original if not found

for wave_num, top_issues in top_issues_by_wave.items():
    # Map features to descriptions
    labeled_issues = map_features_to_labels(top_issues)

    # Save for reporting
    labeled_issues.to_csv(f"wave{wave_num}_top_issues_labeled.csv", index=False)
    print(f"✅ Labeled top issues saved for Wave {wave_num}")

NameError: name 'top_issues_by_wave' is not defined